# 10 Quantum Single Option Pricing

Reconstruct a single option price curve with QFT-style diagonalization, unitary dilation, and post-selection. A small Qiskit prototype is included when available.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
spot = 24000.0; K = 24000.0; T = 30 / 365; r = 0.065; sigma = 0.18; option_type = "put"
q = quantum_price_reconstruction(spot, K, T, r, sigma, option_type, n_qubits=int(config["quantum"]["default_grid_qubits"]), x_width=float(config["quantum"]["x_width"]))
fd = finite_difference_black_scholes_solver(K=K, T=T, r=r, sigma=sigma, option_type=option_type, S_max=3*K, stock_steps=160, time_steps=160)
fd_interp = np.interp(q["S_grid"], fd["S_grid"], fd["price_grid"])
metrics_bs = price_error_metrics(q["classical_curve"], q["price_curve"], q["S_grid"], K, option_type)
metrics_fd = price_error_metrics(fd_interp, q["price_curve"], q["S_grid"], K, option_type)
assert 0 <= q["post_selection_probability"] <= 1
assert np.isfinite(q["price_at_spot"])
print("VALIDATION PASSED: quantum reconstruction produced finite price and valid post-selection probability")
plt.figure()
plt.plot(q["S_grid"], q["classical_curve"], label="Analytical BS")
plt.plot(q["S_grid"], fd_interp, "--", label="Finite difference")
plt.plot(q["S_grid"], q["price_curve"], label="Quantum reconstructed")
plt.axvline(K, color="black", linewidth=1, linestyle="--")
plt.title("Quantum reconstructed price versus classical baselines")
plt.xlabel("Index level")
plt.ylabel("Put value")
plt.legend()
save_current_figure("10_quantum_reconstructed_vs_classical.png")
plt.figure()
plt.plot(q["S_grid"], q["price_curve"] - q["classical_curve"])
plt.axhline(0, color="black", linewidth=1)
plt.title("Quantum pricing error versus stock price")
plt.xlabel("Index level")
plt.ylabel("Quantum - analytical")
save_current_figure("10_quantum_pricing_error.png")
save_table(pd.DataFrame({"S": q["S_grid"], "quantum": q["price_curve"], "black_scholes": q["classical_curve"], "finite_difference": fd_interp}), "10_quantum_single_option_curve.csv")
save_output({"against_black_scholes": metrics_bs, "against_finite_difference": metrics_fd, "post_selection_probability": q["post_selection_probability"], "price_at_spot": q["price_at_spot"], "classical_at_spot": q["classical_at_spot"]}, "10_quantum_single_option_metrics.json")
metrics_bs


In [ ]:
# Small Qiskit circuit prototype. Core project results do not depend on Qiskit availability.
prototype = {"qiskit_available": False}
try:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
    qc = QuantumCircuit(3, 3)
    qc.h(0); qc.h(1); qc.cp(np.pi / 2, 1, 0); qc.h(2)
    qc.barrier()
    qc.rz(0.2, 0); qc.rz(-0.1, 1); qc.rz(0.05, 2)
    qc.barrier()
    qc.measure([0, 1, 2], [0, 1, 2])
    sv_circuit = qc.remove_final_measurements(inplace=False)
    sv = Statevector.from_instruction(sv_circuit)
    counts = sv.sample_counts(512)
    prototype = {"qiskit_available": True, "counts": dict(counts), "depth": int(qc.depth()), "two_qubit_count": int(qc.count_ops().get("cp", 0))}
    print(qc.draw(output="text"))
except Exception as exc:
    prototype = {"qiskit_available": False, "reason": str(exc)}
    print("Qiskit prototype skipped:", exc)
plot_small_circuit_schematic("10_small_qft_circuit_schematic.png")
save_output(prototype, "10_small_qiskit_prototype.json")
prototype
